In [ ]:
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"idabagusmantra","key":"6b3385b6114e0ad29db111259bee0168"}'}

In [ ]:
!mkdir -p /root/.config/kaggle
!mv kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json


In [ ]:
!ls -l /root/.config/kaggle/


total 4
-rw------- 1 root root 70 Jan  7 13:02 kaggle.json


In [ ]:
!kaggle competitions list


ref                                                                                 deadline             category                reward  teamCount  userHasEntered  
----------------------------------------------------------------------------------  -------------------  ---------------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3       2026-04-15 23:59:00  Featured         2,207,152 Usd       1220           False  
https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection            2026-02-13 23:59:00  Research           200,000 Usd        575           False  
https://www.kaggle.com/competitions/google-tunix-hackathon                          2026-01-12 23:59:00  Featured           100,000 Usd        128           False  
https://www.kaggle.com/competitions/csiro-biomass                                   2026-01-28 23:59:00  Research            75,000 Usd       3104           False  
https://ww

In [ ]:
!kaggle competitions download -c playground-series-s6e1
!unzip playground-series-s6e1.zip
!ls


  0% 0.00/13.8M [00:00<?, ?B/s]
100% 13.8M/13.8M [00:00<00:00, 1.29GB/s]
Archive:  playground-series-s6e1.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
'kaggle (1).json'      'kaggle (2).json'	     sample_submission.csv
'kaggle (2) (1).json'   playground-series-s6e1.zip   test.csv
'kaggle (2) (2).json'   sample_data		     train.csv


In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(train.shape)
print(test.shape)
train.head()


(630000, 13)
(270000, 12)


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


In [ ]:
X = train.drop(columns=['exam_score'])
y = train['exam_score']


In [ ]:
X = pd.get_dummies(X)
test = pd.get_dummies(test)

# Samakan kolom train & test
X, test = X.align(test, join='left', axis=1, fill_value=0)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


RandomForestRegressor(max_depth=12, n_estimators=50, n_jobs=-1, random_state=42)

In [ ]:
print(X_train.shape)


(504000, 31)


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    max_depth=8,
    learning_rate=0.1,
    max_iter=200,
    random_state=42
)

model.fit(X_train, y_train)


HistGradientBoostingRegressor(max_depth=8, max_iter=200, random_state=42)

In [ ]:

test_ids = test["id"]

# Buang kolom target (tidak ada di test, tapi kita samakan struktur)
X_test = test.drop(columns=["id"])


In [ ]:
# Pisahkan target
y = train["exam_score"]

# Buang kolom id dan target
X = train.drop(columns=["id", "exam_score"])


In [ ]:
# Target
y = train["exam_score"]

# Drop id & target
X = train.drop(columns=["id", "exam_score"])

# One-hot encoding
X = pd.get_dummies(X)


In [ ]:
test_ids = test["id"]

X_test = test.drop(columns=["id"])
X_test = pd.get_dummies(X_test)

# Samakan kolom test dengan train
X_test = X_test.reindex(columns=X.columns, fill_value=0)


In [ ]:
!pip install lightgbm


In [ ]:
import pandas as pd

y = train["exam_score"]
X = train.drop(columns=["id", "exam_score"])

X = pd.get_dummies(X)


In [ ]:
test_ids = test["id"]
X_test = test.drop(columns=["id"])

X_test = pd.get_dummies(X_test)
X_test = X_test.reindex(columns=X.columns, fill_value=0)


In [ ]:
print(X.shape, X_test.shape)


(630000, 30) (270000, 30)


In [ ]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.106517 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 30
[LightGBM] [Info] Start training from score 62.506672


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, n_estimators=1000,
              n_jobs=-1, random_state=42, subsample=0.8)

In [ ]:
test_pred = model.predict(X_test)


In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "exam_score": test_pred
})

submission.to_csv("submission_lgb.csv", index=False)


In [ ]:
from google.colab import files
files.download("submission_lgb.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import lightgbm as lgb

model_lgb = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_lgb.fit(X, y)

test_pred_lgb = model_lgb.predict(X_test)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.049561 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 30
[LightGBM] [Info] Start training from score 62.506672


In [ ]:
import numpy as np

y_log = np.log1p(y)

model_log = lgb.LGBMRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_log.fit(X, y_log)

test_pred_log = np.expm1(model_log.predict(X_test))


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.049412 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 30
[LightGBM] [Info] Start training from score 4.100145


In [ ]:
max_bin=255


In [ ]:
final_pred = 0.5 * test_pred_lgb + 0.5 * test_pred_log


In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "exam_score": final_pred
})

submission.to_csv("submission_blend.csv", index=False)
from google.colab import files
files.download("submission_blend.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.model_selection import KFold
import numpy as np
import lightgbm as lgb

kf = KFold(n_splits=5, shuffle=True, random_state=42)

test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)
    test_preds += model.predict(X_test) / 5


Fold 1
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.039019 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 622
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.482335
Fold 2
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.037775 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 622
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.502155
Fold 3
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-thread

In [ ]:
test_preds = np.clip(test_preds, 0, 100)


In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "exam_score": test_preds
})

submission.to_csv("submission_kfold.csv", index=False)
from google.colab import files
files.download("submission_kfold.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>